In [39]:
import requests
from bs4 import BeautifulSoup

In [67]:
url = 'https://news.google.com/rss/headlines/section/topic/BUSINESS'
response = requests.get(url)
soup = BeautifulSoup(response.content, 'xml')

In [68]:
headlines = soup.find('channel').find_all('title')

In [69]:
hlist = []
headlines = soup.find('channel').find_all('title')
for item in headlines:
    hlist.append(item.get_text())
del hlist[0:2]

In [62]:
!pip install --quiet torch
import torch
print(torch.__version__)

2.10.0+cpu


In [63]:
!pip install --quiet transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from scipy.special import softmax

In [64]:
import warnings
warnings.filterwarnings('ignore')
source_model = f"ProsusAI/finbert"
tokenizer = AutoTokenizer.from_pretrained(source_model)
model = AutoModelForSequenceClassification.from_pretrained(source_model)

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [118]:
example_text = hlist[1]
example_text

'Dow futures jump 900 points, oil tumbles after Trump suspends Iran attacks for two weeks: Live updates - CNBC'

In [119]:
encoded_text = tokenizer(example_text, return_tensors = 'pt')
encoded_text

{'input_ids': tensor([[  101, 23268, 17795,  5376,  7706,  2685,  1010,  3514, 28388,  2015,
          2044,  8398, 28324,  2015,  4238,  4491,  2005,  2048,  3134,  1024,
          2444, 14409,  1011, 27166,  9818,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1]])}

In [120]:
output = model(**encoded_text)
output

SequenceClassifierOutput(loss=None, logits=tensor([[-1.3474,  0.6723,  1.5191]], grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)

In [121]:
scores = output[0][0].detach().numpy()
scores = softmax(scores)
scores

array([0.03829584, 0.2886046 , 0.6730995 ], dtype=float32)

In [122]:
def get_sentiment(example):
    encoded_text = tokenizer(example, return_tensors = 'pt')
    output = model(**encoded_text)
    scores = output[0][0].detach().numpy()
    scores = softmax(scores)
    scores_dict = {
        'finbert_neg' : scores[0],
        'finbert_neu' : scores[1],
        'finbert_pos' : scores[2]
    }
    return scores_dict

In [123]:
get_sentiment(hlist[1])

{'finbert_neg': np.float32(0.038295835),
 'finbert_neu': np.float32(0.2886046),
 'finbert_pos': np.float32(0.6730995)}

In [124]:
import pandas as pd
headlines_df = pd.DataFrame(hlist, columns = ['Headlines'])

In [125]:
hl_sentiment = headlines_df['Headlines'].apply(get_sentiment)

In [131]:
finbert_neg = []
for row in hl_sentiment:
    finbert_neg.append(list(row.values())[0])
finbert_neut = []
for row in hl_sentiment:
    finbert_neut.append(list(row.values())[1])
finbert_pos = []
for row in hl_sentiment:
    finbert_pos.append(list(row.values())[2])

In [132]:
final_hl_df = pd.DataFrame({
    'Headline': hlist,
    'Negative finBERT Score': finbert_neg,
    'Neutral finBERT Score': finbert_neut,
    'Positive finBERT Score': finbert_pos
})
    

In [133]:
final_hl_df

,Headline,Negative finBERT Score,Neutral finBERT Score,Positive finBERT Score
0,Anthropic debuts preview of powerful new AI mo...,0.192942,0.009460,0.797598
1,"Dow futures jump 900 points, oil tumbles after...",0.038296,0.288605,0.673100
2,Gas Prices Are Americans’ Top Concern in Iran ...,0.045970,0.036056,0.917974
3,Delta joins the growing list of US airlines ra...,0.427543,0.052248,0.520208
4,Southwest Airlines to limit passengers to one ...,0.033259,0.057877,0.908864
5,Doritos prices jumped 50% in four years and Pe...,0.214120,0.022184,0.763696
6,Key Fed official sees possible rate hike amid ...,0.255769,0.118020,0.626211
7,Jet fuel supply concerns grow as war with Iran...,0.011357,0.935112,0.053530
8,Deere & Co agrees to pay $99 million to settle...,0.099371,0.053323,0.847306
9,Bill Ackman’s Pershing Square Bids to Buy Univ...,0.095985,0.012284,0.891731
